In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
catalog_name = 'ecommerce'

### Brands

In [0]:
# Define schema for the data file
brand_schema = StructType([
    StructField('brand_code', StringType(), False),
    StructField('brand_name', StringType(), True),
    StructField('category_code', StringType(), True)
])

raw_data_path = '/Volumes/ecommerce/source_data/raw/brands/*.csv'

df = spark.read.option('header','true').option('delimiter',',').schema(brand_schema).csv(raw_data_path)

df = df.withColumn('_source_file', col('_metadata.file_path'))\
    .withColumn('ingested_at', current_timestamp())

display(df.limit(5))    

df.write.format('delta').mode('overwrite').option('mergeSchema','true').saveAsTable(f'{catalog_name}.bronze.brz_brands')

print('Brands table has been successfully created in the Bronze layer.')

### Category

In [0]:
category_schema = StructType([
    StructField('category_code', StringType(), False),
    StructField('category_name', StringType(), True)
])

raw_data_path = '/Volumes/ecommerce/source_data/raw/category/category.csv'

df = spark.read.option('header','true').option('delimiter',',').schema(category_schema).csv(raw_data_path)

df = df.withColumn('_source_file', col('_metadata.file_path'))\
    .withColumn('ingested_at', current_timestamp())

display(df.limit(5))   

df.write.format('delta').mode('overwrite').option('mergeSchema','true').saveAsTable(f'{catalog_name}.bronze.brz_category')

print('Category table has been successfully created in the Bronze layer.')

### Customers

In [0]:
customers_schema = StructType([
    StructField('customer_id', StringType(), False),
    StructField('phone', StringType(), True),
    StructField('country_code', StringType(), True),
    StructField('country',StringType(), True),
    StructField('state',StringType(), True)
])

raw_data_path = '/Volumes/ecommerce/source_data/raw/customers/customers.csv'

df = spark.read.option('header','true').option('delimiter',',').schema(customers_schema).csv(raw_data_path)

df = df.withColumn('_source_file', col('_metadata.file_path'))\
    .withColumn('ingested_at', current_timestamp())

display(df.limit(5))

df.write.format('delta').mode('overwrite').option('mergeSchema','true').saveAsTable(f'{catalog_name}.bronze.brz_customer')

print('Customer table has been successfully created in the Bronze layer.')

### Products

In [0]:
products_schema = StructType([
    StructField('product_id', StringType(), False),
    StructField('sku', StringType(), True),
    StructField('category_code', StringType(), True),
    StructField('brand_code', StringType(), True),
    StructField('color', StringType(), True),
    StructField('size', StringType(), True),
    StructField('material', StringType(), True),
    StructField('weight_grams', StringType(), True),  
    StructField('length_cm', StringType(), True),    
    StructField('width_cm', FloatType(), True),
    StructField('height_cm', FloatType(), True),
    StructField('rating_count', IntegerType(), True),
    StructField('file_name', StringType(), False),
    StructField('ingest_timestamp', TimestampType(), False)
])

raw_data_path = '/Volumes/ecommerce/source_data/raw/products/products.csv'

df = spark.read.option('header', 'true').option('delimiter', ',').schema(products_schema).csv(raw_data_path).withColumn('file_name', col('_metadata.file_path')).withColumn('ingest_timestamp', current_timestamp())

df.write.format('delta').mode('overwrite').option('mergeSchema', 'true').saveAsTable(f'{catalog_name}.bronze.brz_products')    

display(df.limit(5))

print('Products table has been successfully created in the Bronze layer.')

### Date

In [0]:
date_schema = StructType([
    StructField('date', StringType(), True),           
    StructField('year', IntegerType(), True),         
    StructField('day_name', StringType(), True),       
    StructField('quarter', IntegerType(), True),       
    StructField('week_of_year', IntegerType(), True),  
])

raw_data_path = f'/Volumes/ecommerce/source_data/raw/date/date.csv' 

df_raw = spark.read.option('header', 'true').option('delimiter', ',').schema(date_schema).csv(raw_data_path)

df_raw = df_raw.withColumn('_ingested_at', current_timestamp()).withColumn('_source_file', col('_metadata.file_path'))


df_raw.write.format('delta').mode('overwrite').option('mergeSchema', 'true').saveAsTable(f'{catalog_name}.bronze.brz_calendar')    

display(df.limit(5))

print('Date table has been successfully created in the Bronze layer.')           